# 1D advection-dispersion with first-order kinetic decay

Checks how mf6rtm passes the time step to PhreeqcRM for KINETICS.
A neutral tracer `Tr` decays as $dC/dt = -kC$ (PHREEQC `KINETICS` + `RATES`). The column starts at
$C_i$ and is flushed with $C_0$ through a flux inlet. References:

1. analytical solution, van Genuchten & Alves (1982): semi-infinite column, flux inlet,
   first-order decay, uniform initial concentration;
2. PHREEQC `TRANSPORT`, same database and rates (`decay1d.pqi`, results in `decay1d.sel`;
   rerun with `phreeqc decay1d.pqi decay1d.out decay1d_datab.dat`).

Ahead of the front the column stays uniform, so transport does nothing there and the exact answer
is $C_i e^{-kt}$. That zone tests the reaction time step alone.

mf6rtm is run twice with its default settings: constant $\Delta t$ = 1 d (= PHREEQC shift time),
and a growing $\Delta t$ (`tsmult` = 1.1).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import flopy
from scipy.special import erfc, erfcx
from mf6rtm import mup3d, utils

HERE = Path.cwd()                 # run the notebook from benchmark/
BIN =  'bin'
FOLDER = HERE / 'decay1d'

## Parameters (same as `decay1d.pqi`)

In [ ]:
ncol, dx = 100, 0.1          # 10 m column
n = 0.3                      # porosity
v = 0.1                      # pore velocity (m/d)
alpha = 0.05                 # longitudinal dispersivity (m)
D = alpha * v                # m2/d
k = 0.03                     # first-order decay (1/d)
C0, Ci = 1e-3, 5e-4          # inflow / initial Tr (mol/kgw)
T = 60.0                     # d
x = (np.arange(ncol) + 0.5) * dx
x_uniform = 9.05             # uniform zone until the front tail arrives (~40 d)

RUNS = {                     # label: (run folder, MF6 TDIS (perlen, nstp, tsmult))
    FOLDER / 'mf6rtm, dt = 1 d': ('run_const', (T, 60, 1.0)),
    FOLDER / 'mf6rtm, tsmult 1.1': ('run_tsmult', (T, 20, 1.1)),   # dt from 1.05 to 6.45 d
}

## Analytical solution (van Genuchten & Alves, 1982)

In [ ]:
def _e_erfc(a, z):
    """exp(a) * erfc(z) without overflow."""
    z = np.asarray(z, float)
    return np.where(z > 0, np.exp(a - z**2) * erfcx(np.abs(z)), np.exp(a) * erfc(z))


def analytical(x, t):
    s = 2 * np.sqrt(D * t)
    u = v * np.sqrt(1 + 4 * k * D / v**2)
    A = (v / (v + u) * _e_erfc((v - u) * x / (2 * D), (x - u * t) / s)
         + v / (v - u) * _e_erfc((v + u) * x / (2 * D), (x + u * t) / s)
         + v**2 / (2 * k * D) * _e_erfc(v * x / D - k * t, (x + v * t) / s))
    B = (1 - 0.5 * erfc((x - v * t) / s)
         - np.sqrt(v**2 * t / (np.pi * D)) * np.exp(-(x - v * t)**2 / (4 * D * t))
         + 0.5 * (1 + v * x / D + v**2 * t / D) * _e_erfc(v * x / D, (x + v * t) / s))
    return C0 * A + Ci * np.exp(-k * t) * B

## PHREEQC TRANSPORT results

In [ ]:
phq = pd.read_csv(FOLDER / 'phreeqc' / 'decay1d.sel', sep=r'\s+')
phq = phq[phq.cell > 0].rename(columns={'time_d': 't'})

## mf6rtm runs

In [ ]:
def run_mf6rtm(folder, tdis):
    sol = mup3d.Solutions({'pH': [7.0, 7.0], 'Na': [1e-3, 1e-3], 'Cl': [1e-3, 1e-3], 'Tr': [Ci, C0]})
    sol.set_ic(np.ones((1, 1, ncol), dtype=int))
    kin = mup3d.KineticPhases({1: {'Decay': {'m0': 1.0, 'parms': [k / 86400], 'formula': 'Tr 1'}}})
    kin.set_ic(np.ones((1, 1, ncol), dtype=int))
    model = mup3d.Mup3d('decay', sol, 1, 1, ncol)
    model.set_wd(str(FOLDER / folder))
    model.set_database(str(FOLDER /'phreeqc' / 'decay1d_datab.dat'))
    model.set_phases(kin)
    model.set_postfix(str(FOLDER / 'phreeqc' /'decay1d_postfix.phqr'))
    model.initialize()
    wel = mup3d.ChemStress('wel')
    wel.set_spd([2])
    model.set_chem_stress(wel)

    sim = flopy.mf6.MFSimulation(sim_name='decay', sim_ws=str(model.wd), exe_name='mf6')
    flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[tdis], time_units='days')
    gwf = flopy.mf6.ModflowGwf(sim, modelname='gwf')
    sim.register_ims_package(flopy.mf6.ModflowIms(sim, filename='gwf.ims'), ['gwf'])
    dis = dict(nlay=1, nrow=1, ncol=ncol, delr=dx, delc=1.0, top=1.0, botm=0.0)
    flopy.mf6.ModflowGwfdis(gwf, **dis)
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=1.0)
    flopy.mf6.ModflowGwfic(gwf, strt=1.0)
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=[[(0, 0, ncol - 1), 1.0]])
    flopy.mf6.ModflowGwfwel(gwf, stress_period_data=[[(0, 0, 0), v * n] + list(model.wel.data[0])],
                            auxiliary=model.components, pname='wel')
    for c in model.components:
        gwt = flopy.mf6.MFModel(sim, model_type='gwt6', modelname=c, model_nam_file=f'{c}.nam')
        ims = flopy.mf6.ModflowIms(sim, linear_acceleration='BICGSTAB', outer_dvclose=1e-10,
                                   inner_dvclose=1e-10, rcloserecord=1e-10, filename=f'{c}.ims')
        sim.register_ims_package(ims, [c])
        flopy.mf6.ModflowGwtdis(gwt, **dis)
        flopy.mf6.ModflowGwtic(gwt, strt=model.sconc[c])
        flopy.mf6.ModflowGwtssm(gwt, sources=[['wel', 'aux', c]])
        flopy.mf6.ModflowGwtadv(gwt, scheme='tvd')
        flopy.mf6.ModflowGwtdsp(gwt, xt3d_off=True, alh=alpha, ath1=alpha)
        flopy.mf6.ModflowGwtmst(gwt, porosity=n)
        flopy.mf6.ModflowGwfgwt(sim, exgtype='GWF6-GWT6', exgmnamea='gwf', exgmnameb=c,
                                filename=f'{c}.gwfgwt')
    sim.write_simulation(silent=True)
    utils.prep_bins(str(model.wd), src_path=str(BIN))
    model.run()

    out = pd.read_csv(Path(model.wd) / 'sout.csv')
    # time at the end of each step, from TDIS
    per, nstp, mult = tdis
    dts = np.full(nstp, per / nstp) if mult == 1 else per * (mult - 1) / (mult**nstp - 1) * mult**np.arange(nstp)
    out['t'] = np.repeat(np.cumsum(dts), ncol)
    return out


res = {name: run_mf6rtm(*args) for name, args in RUNS.items()}

## Results

Left: profile at $t$ = 60 d. Right: the uniform zone, where the exact answer is $C_i e^{-kt}$.
PHREEQC itself is ~1.5 % low behind the front; that's the sequential operator-splitting error
($\approx e^{-k\Delta t/2} - 1$), not a bug. The uniform zone has no splitting error.

In [ ]:
icell = int(np.argmin(np.abs(x - x_uniform))) + 1
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(x, analytical(x, T) * 1e3, 'k-', lw=2, label='analytical')
s = phq[np.isclose(phq.t, T)].sort_values('cell')
ax1.plot(x, s.Tr * 1e3, 'k:', lw=2, label='PHREEQC TRANSPORT')
for name, d in res.items():
    s = d[np.isclose(d.t, T)].sort_values('cell')
    ax1.plot(x, s.Tr * 1e3, '--', label=str(name).split('/')[-1])
ax1.set(xlabel='x (m)', ylabel='Tr (mmol/kgw)', title=f't = {T:g} d')
ax1.legend()

tt = np.linspace(0, T, 200)
ax2.plot(tt, Ci * np.exp(-k * tt) * 1e3, 'k-', lw=2, label=r'exact, $C_i e^{-kt}$')
s = phq[phq.cell == icell]
ax2.plot(s.t, s.Tr * 1e3, 'k:', lw=2, label='PHREEQC TRANSPORT')
for name, d in res.items():
    s = d[d.cell == icell]
    ax2.plot(s.t, s.Tr * 1e3, 'o--', ms=3, label=str(name).split('/')[-1])
ax2.set(xlabel='t (d)', ylabel='Tr (mmol/kgw)', title=f'uniform zone, x = {x[icell - 1]:.2f} m')
ax2.legend()
fig.tight_layout()

Relative error in the uniform zone (front still ≥ 5 m away for $t \le$ 40 d):

In [ ]:
def uniform_err(d):
    s = d[(d.cell == icell) & (d.t > 0) & (d.t <= 40.0)]
    return np.max(np.abs(s.Tr / (Ci * np.exp(-k * s.t)) - 1))


errs = {'PHREEQC TRANSPORT': uniform_err(phq)} | {str(name).split('/')[-1]: uniform_err(d) for name, d in res.items()}
pd.Series(errs, name='max rel. error, uniform zone, t ≤ 40 d').map('{:.2%}'.format)